In [ ]:
import os
import glob
import itertools
import h5py
import numpy as np
import matplotlib.pyplot as plt
import gvar as gv
import lsqfit

#======== what to analyse ========
hadron = "pion"              #"pion" or "proton"
frame = "sym"                #"forward" or "sym"
q = (0,-2,0)                 #the q of a cell that does not average over q;  forward: (0,0,0);  sym: (2,0,0), (-2,0,0), (0,2,0) or (0,-2,0)
project = True               #proton only: True = positive-parity files, False = open Dirac files (none on this laptop)
ncfg = 250
Lt = 96
Ls = 32
hyp = 8                      #the HYP step of the gluon operator
phase_attached_to = "2pt"    #vacuum term: "FF" = zero to rounding at q != 0 (old build_3pt.py), "2pt" = noise x noise;  the same at q = 0

#======== which saved GEVP basis cells 2, 4 and 5 use ========
#an operator is (smear, gamma);  smear 0 = B [4.5,2.25,0.8], 1 = U [1.5,1.5,0];  gamma 0 = G5, 1 = G5G4, 2 = G3G5
op_basis = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]
t0 = 1                       #GEVP reference time
t_ref = 3                    #time of the ground-state vector v0
smear_names = ["B", "U"]
gamma_names = ["G5", "G5G4", "G3G5"]

#======== folders ========
pt2_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta"
save_dir = os.path.join(pt2_dir, "AI_assisted_matrix_element_analysis")
#C3input: q-phase averaged in;  C2pf, C2pi: no q-phase averaged in;  at q = 0 the C3input file is also the C2pf and the C2pi file
pt2_template = os.path.join(pt2_dir, "2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_{frame}_q{qx}_{qy}_{qz}_{pt2_type}_fb_tsep20_3072src_ncfg{ncfg}_{hadron}.h5")

#======== what follows from the settings above ========
assert frame in ["forward", "sym"], f"frame = {frame}: use forward or sym"
if frame == "forward":
    q_list = [(0,0,0)]
else:
    q_list = [(2,0,0), (-2,0,0), (0,2,0), (0,-2,0)]
q = tuple(q)
assert q in q_list, f"frame = {frame} with q = {q}: use one of {q_list}"
assert t_ref > t0, f"t0 = {t0}, t_ref = {t_ref}: t_ref has to be larger than t0"

if hadron == "pion":
    hadron_tag = "pion"
elif project:
    hadron_tag = "proton_projected"
else:
    hadron_tag = "proton"
c_sign = {"pion": -1, "proton": +1}[hadron]      #to make Cij positive definite, the pion is multiplied by -1

with h5py.File(pt2_template.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag), "r") as f:
    smear_list = f["smear_list"][()]             #(smear, 3): rho_T, rho_z, boost fraction


#======== functions used by several cells ========
def jk(data, jk_axis=0):
    #jackknife mean and error
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    #per-cfg data (cfg, ...) -> jackknife samples (jk, ...)
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def read_pt2(q_i, pt2_type):
    #both halves of a packed 2pt file: (cfg, smear, smear, gamma, gamma, t_src, pz, tsep)
    pt2_file = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type=pt2_type, ncfg=ncfg, hadron=hadron_tag)
    print("reading", pt2_file)
    assert os.path.exists(pt2_file), f"no such file: {pt2_file}"
    with h5py.File(pt2_file, "r") as f:
        return f["pt2_forward"][()], f["pt2_backward"][()]


def gevp_name(basis):
    #the operator labels, the tag and the GEVP file of a basis;  the tag goes in the name of every saved file
    labels = [smear_names[s] + gamma_names[g] for s, g in basis]     #["BG5", "BG5G4", ...]
    tag = "_".join(labels) + f"_t0{t0}_tref{t_ref}"
    basis_file = os.path.join(save_dir, "2pt_gevp_data", frame, f"2pt_gevp_{frame}_{tag}_ncfg{ncfg}_{hadron_tag}.h5")
    return labels, tag, basis_file


def save_gevp_info(f, basis):
    #what every saved file states about the GEVP;  f is an open h5py file
    labels, tag, basis_file = gevp_name(basis)
    f["gevp_tag"] = tag
    f["gevp_basis"] = labels            #(op,): the operators in words
    f["op_basis"] = basis               #(op, 2): (smear, gamma) indices of smear_names, gamma_names
    f["smear_names"] = smear_names
    f["smear_list"] = smear_list
    f["gamma_names"] = gamma_names
    f["t0"] = t0
    f["t_ref"] = t_ref


def op_matrix(pt2, basis):
    #(cfg, smear_snk, smear_src, gamma_snk, gamma_src, ...) -> (cfg, op_snk, op_src, ...)
    n_op = len(basis)
    Cij = np.zeros(pt2.shape[:1] + (n_op, n_op) + pt2.shape[5:], dtype=complex)
    for a, (s_snk, g_snk) in enumerate(basis):
        for b, (s_src, g_src) in enumerate(basis):
            Cij[:, a, b] = pt2[:, s_snk, s_src, g_snk, g_src]
    return Cij


def apply_v0(Cij, v):
    #v^dagger C v:  Cij (cfg, op, op, ..., pz, tsep), v (pz, op) -> (cfg, ..., pz, tsep)
    return c_sign * np.einsum("pa,cab...pT,pb->c...pT", v.conj(), Cij, v)


def pt3_path(q_i):
    #the 3pt file of the basis of this cell at one q
    q_tag_i = f"q{q_i[0]}_{q_i[1]}_{q_i[2]}"
    if frame == "forward":
        folder = "forward"
    else:
        folder = f"sym_{q_tag_i}"
    return os.path.join(save_dir, "3pt_data", folder, f"3pt_{frame}_{q_tag_i}_{gevp_tag}_phase{phase_attached_to}_hyp{hyp}_ncfg{ncfg}_{hadron_tag}.h5")


op_labels, gevp_tag, gevp_file = gevp_name(op_basis)
print("hadron", hadron_tag, "| frame", frame, "| q", q)
print("GEVP basis", " ".join(op_labels), "| t0", t0, "| t_ref", t_ref)

In [ ]:
#======== GEVP: reconstructed 2pt and v0 of many bases, one file per basis ========
#run once per hadron, frame, t0 and t_ref (cell 0 sets them);  the files stay on disk
gevp_all_ops = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]    #the operators the bases are made from, (smear, gamma) as in cell 0
gevp_basis_sizes = [1,2,3,4,5,6]                        #how many operators in a basis;  1 to 6 gives all 63 bases


def solve_gevp(C_matrix):
    #C_matrix (op, op, pz, tsep): eigenvalues (pz, state, tsep) and eigenvectors (pz, state, tsep, op), largest eigenvalue first, NaN at tsep <= t0
    n_op, _, n_pz, n_tsep = C_matrix.shape
    eigen_val = np.full((n_pz, n_op, n_tsep), np.nan)
    eigen_vec = np.full((n_pz, n_op, n_tsep, n_op), np.nan, dtype=complex)
    for ipz in range(n_pz):
        L_inv = np.linalg.inv(np.linalg.cholesky(C_matrix[:, :, ipz, t0]))
        for t in range(t0 + 1, n_tsep):
            M = L_inv @ C_matrix[:, :, ipz, t] @ L_inv.conj().T
            w, u = np.linalg.eigh(M)
            v = L_inv.conj().T @ u
            order = np.argsort(w)[::-1]
            eigen_val[ipz, :, t] = w[order]
            eigen_vec[ipz, :, t] = v[:, order].T
    return eigen_val, eigen_vec


#---- read the 2pt of every q and average over t_src
if frame == "forward":
    pt2_type = "C3input"        #at q = 0 this file is also C2pf and C2pi
else:
    pt2_type = "C2pf"           #C2pi(q) is exactly C2pf(-q), so the C2pi files are not needed

half_list = ["forward", "backward"]
C2_raw = {"forward": [], "backward": []}    #C2_raw[half][iq]: (cfg, smear, smear, gamma, gamma, pz, tsep)
pf_list = []                                #pf_list[iq]: (pz, 3), the momentum of each 2pt
for q_i in q_list:
    raw_forward, raw_backward = read_pt2(q_i, pt2_type)
    assert raw_forward.ndim == 8, "the GEVP needs (cfg, smear, smear, gamma, gamma, t_src, pz, tsep): open Dirac files have to be projected first"
    C2_raw["forward"].append(raw_forward.mean(axis=5))
    C2_raw["backward"].append(raw_backward.mean(axis=5))
    pt2_file_i = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type=pt2_type, ncfg=ncfg, hadron=hadron_tag)
    with h5py.File(pt2_file_i, "r") as f:
        pf_list.append(f["pf_list"][()])
        pz_list = f["pz_list"][()]
        if q_i == q_list[0]:
            cfg_list = f["cfg_list"][()]
        assert np.array_equal(f["cfg_list"][()], cfg_list), f"q = {q_i}: other cfgs than the first q"

n_cfg = C2_raw["forward"][0].shape[0]
n_pz = C2_raw["forward"][0].shape[5]
n_tsep = C2_raw["forward"][0].shape[6]
n_q = len(q_list)

#---- every basis: solve the GEVP, reconstruct the 2pt, save
n_saved = 0
for n_op in gevp_basis_sizes:
    for basis in itertools.combinations(gevp_all_ops, n_op):
        basis = list(basis)
        labels, tag, basis_file = gevp_name(basis)
        C2_rec = {}
        v0 = {}
        eigen_val = {}
        for half in half_list:
            C2_rec[half] = np.zeros((n_cfg, n_q, n_pz, n_tsep), dtype=complex)      #(cfg, q, pz, tsep)
            v0[half] = np.zeros((n_q, n_pz, n_op), dtype=complex)                   #(q, pz, op)
            eigen_val[half] = np.zeros((n_q, n_pz, n_op, n_tsep))                   #(q, pz, state, tsep)
            for iq, q_i in enumerate(q_list):
                iq_minus = q_list.index((-q_i[0], -q_i[1], -q_i[2]))
                C_plus = op_matrix(C2_raw[half][iq], basis)                         #(cfg, op, op, pz, tsep)
                C_minus = op_matrix(C2_raw[half][iq_minus], basis)                  #(cfg, op, op, pz, tsep)
                #the mean over cfg of [C2(q) + C2(-q)] / 2, positive sign, hermitian part: (op, op, pz, tsep)
                C_mean = c_sign * 0.5 * (C_plus + C_minus).mean(axis=0)
                C_mean = 0.5 * (C_mean + C_mean.conj().swapaxes(0, 1))
                eigen_val[half][iq], eigen_vec = solve_gevp(C_mean)                 #eigen_vec (pz, state, tsep, op);  state 0 is the ground state
                v0[half][iq] = eigen_vec[:, 0, t_ref, :]
                C2_rec[half][:, iq] = apply_v0(C_plus, v0[half][iq])

        os.makedirs(os.path.dirname(basis_file), exist_ok=True)
        with h5py.File(basis_file, "w") as f:
            for half in half_list:
                f[f"C2_{half}"] = C2_rec[half]              #(cfg, q, pz, tsep), complex, c_sign already in;  q in the order of q_list
                f[f"v0_{half}"] = v0[half]                  #(q, pz, op)
                f[f"eigen_val_{half}"] = eigen_val[half]    #(q, pz, state, tsep), NaN at tsep <= t0
            f["q_list"] = q_list                            #(q, 3)
            f["pf_list"] = np.array(pf_list)                #(q, pz, 3): the momentum of C2
            f["pz_list"] = pz_list
            f["cfg_list"] = cfg_list
            save_gevp_info(f, basis)
        n_saved = n_saved + 1

print("saved", n_saved, "files in", os.path.dirname(gevp_file))

In [ ]:
#======== 2pt fit: every fit parameter is here ========
q_average_or_not = True             #True = fit the average of the four q;  False = see single_q_data;  forward ignores this
single_q_data = "pair"              #only when q_average_or_not = False:  "pair" = average of q and -q,  "one" = q only;  q comes from cell 0
pt2_fit_half = "fb"                 #"forward", "backward" or "fb"
pt2_fit_window = {0: (5, 15), 1: (5, 15), 2: (5, 15), 3: (5, 15), 4: (5, 15), 5: (5, 12), 6: (5, 10)}    #pz: (first t, last t)
n_state = 2                         #number of states in the model
pt2_E0_center = "lattice"           #center of the E0 prior:  "lattice": cosh E = cosh m + sum(1 - cos p_i);  "continuum": E^2 = m^2 + sum(p_i^2);  p_i = 2 pi n_i / Ls
pt2_m_rest = None                   #rest mass m above:  None = from the effective mass (next two lines);  a number = use that number
pt2_meff_pz = 0                     #the pz where that effective mass is measured
pt2_meff_t = 10                     #the t of log[C(t) / C(t + 1)] for that effective mass
pt2_E0_width = 0.1                  #width of the E0 prior, the same at every pz
pt2_dE_prior = (0.5, 0.4)           #(center, width) of every gap at every pz;  the fit varies log(dE), so dE stays positive
pt2_c_prior = (0.0, 1e4)            #(center, width) of every c_n
pt2_backward_wave = True            #True = keep the term e^{-E (Lt - t)} in the model
pt2_correlated = True               #True = full covariance;  False = only its diagonal
pt2_svdcut = 1e-7
pt2_maxit = 20000
pt2_tol = 1e-8                      #stopping tolerance of the fit
pt2_fitter = "scipy_least_squares"  #the minimizer
pt2_fit_label = ""                  #extra text in the file name;  set it when you change a prior, svdcut, ..., or the old file is replaced

#---- check the settings
print("GEVP basis", " ".join(op_labels), "| t0", t0, "| t_ref", t_ref)
assert pt2_fit_half in ["forward", "backward", "fb"], f"pt2_fit_half = {pt2_fit_half}: use forward, backward or fb"
assert single_q_data in ["pair", "one"], f"single_q_data = {single_q_data}: use pair or one"
assert pt2_E0_center in ["lattice", "continuum"], f"pt2_E0_center = {pt2_E0_center}: use lattice or continuum"
assert n_state >= 1, f"n_state = {n_state}: use at least 1"
assert os.path.exists(gevp_file), f"no GEVP file for {gevp_tag}: run cell 1 first"

#---- read the reconstructed 2pt of this basis
C2_saved = {}
with h5py.File(gevp_file, "r") as f:
    C2_saved["forward"] = f["C2_forward"][()].real      #(cfg, q, pz, tsep)
    C2_saved["backward"] = f["C2_backward"][()].real    #(cfg, q, pz, tsep)
    pf_saved = f["pf_list"][()]                         #(q, pz, 3)
    pz_list = f["pz_list"][()]
    cfg_list = f["cfg_list"][()]
for pz in pz_list:
    assert pz in pt2_fit_window, f"pz = {pz} is missing from pt2_fit_window"

#---- which q go in, and the label of this fit
iq = q_list.index(q)
iq_minus = q_list.index((-q[0], -q[1], -q[2]))
if frame == "forward":
    iq_use = [0]
    pt2_q_tag = "q0_0_0"
    pt2_subfolder = "forward"
elif q_average_or_not:
    iq_use = [0, 1, 2, 3]
    pt2_q_tag = "averaged_over_q"
    pt2_subfolder = "sym_averaged_over_q"
elif single_q_data == "pair":
    iq_use = [iq, iq_minus]
    pt2_q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
    pt2_subfolder = f"sym_{pt2_q_tag}"
else:
    iq_use = [iq]
    pt2_q_tag = f"q{q[0]}_{q[1]}_{q[2]}_one"
    pt2_subfolder = f"sym_{pt2_q_tag}"

#---- the data of the fit: average over the halves and the q of each cfg, then jackknife
if pt2_fit_half == "fb":
    C2_half = 0.5 * (C2_saved["forward"] + C2_saved["backward"])        #(cfg, q, pz, tsep)
else:
    C2_half = C2_saved[pt2_fit_half]                                    #(cfg, q, pz, tsep)
pt2_fit_jk = jackknife(C2_half[:, iq_use].mean(axis=1))                 #(jk, pz, tsep)

#---- center of the E0 prior
p_i = 2 * np.pi * pf_saved[iq_use[0]] / Ls                              #(pz, 3)
sum_cos = (1 - np.cos(p_i)).sum(axis=1)                                 #(pz,)
sum_p2 = (p_i ** 2).sum(axis=1)                                         #(pz,)
ipz_meff = list(pz_list).index(pt2_meff_pz)
if pt2_m_rest is None:
    C_meff = pt2_fit_jk[:, ipz_meff].mean(axis=0)                       #(tsep,)
    E_meff = np.log(C_meff[pt2_meff_t] / C_meff[pt2_meff_t + 1])
    if pt2_E0_center == "lattice":
        m_rest = np.arccosh(np.cosh(E_meff) - sum_cos[ipz_meff])
    else:
        m_rest = np.sqrt(E_meff ** 2 - sum_p2[ipz_meff])
else:
    m_rest = pt2_m_rest
if pt2_E0_center == "lattice":
    E0_center = np.arccosh(np.cosh(m_rest) + sum_cos)                   #(pz,)
else:
    E0_center = np.sqrt(m_rest ** 2 + sum_p2)                           #(pz,)


#---- the model and the prior
def pt2_model(t, p):
    #sum over the states of c_n (e^{-E_n t} + e^{-E_n (Lt - t)}),  E_n = E0 + dE_1 + ... + dE_n
    E = p["E0"]
    corr = 0
    for n in range(n_state):
        if n > 0:
            E = E + p["dE"][n - 1]
        if pt2_backward_wave:
            wave = gv.exp(-E * t) + gv.exp(-E * (Lt - t))
        else:
            wave = gv.exp(-E * t)
        corr = corr + p["c"][n] * wave
    return corr


def pt2_prior(ipz):
    prior = gv.BufferDict()
    prior["E0"] = gv.gvar(E0_center[ipz], pt2_E0_width)
    prior["c"] = gv.gvar(np.full(n_state, float(pt2_c_prior[0])), np.full(n_state, float(pt2_c_prior[1])))
    if n_state > 1:
        prior["log(dE)"] = gv.log(gv.gvar(np.full(n_state - 1, float(pt2_dE_prior[0])), np.full(n_state - 1, float(pt2_dE_prior[1]))))
    return prior


#---- one fit per pz: first the jackknife mean, then every jackknife sample with the same covariance and prior, started from the fit of the mean
n_jk = pt2_fit_jk.shape[0]
n_pz = len(pz_list)
pt2_E_jk = np.zeros((n_jk, n_pz, n_state))          #(jk, pz, state): E_n
pt2_c_jk = np.zeros((n_jk, n_pz, n_state))          #(jk, pz, state): c_n
pt2_E_central = np.zeros((n_pz, n_state))           #(pz, state): the fit of the jackknife mean
pt2_E_lsqfit_err = np.zeros((n_pz, n_state))        #(pz, state): its lsqfit error, which sees the priors
pt2_c_central = np.zeros((n_pz, n_state))           #(pz, state)
pt2_c_lsqfit_err = np.zeros((n_pz, n_state))        #(pz, state)
pt2_chi2dof = np.zeros(n_pz)                        #(pz,): the fit of the jackknife mean
pt2_Q = np.zeros(n_pz)                              #(pz,)
pt2_fit_result = {}
pt2_n_not_converged = 0                             #how many fits stopped at pt2_maxit before converging;  lsqfit gives no warning for this
gv.switch_gvar()        #the fits keep their covariances apart and gv.restore_gvar() below frees them:  without this, every rerun of this cell takes more memory and time
for ipz, pz in enumerate(pz_list):
    first, last = pt2_fit_window[pz]
    t = np.arange(first, last + 1)                  #(n_t,)
    assert len(t) > n_state + 1, f"pz = {pz}: {len(t)} points for {n_state + 1} free parameters, make the window longer"
    y_jk = pt2_fit_jk[:, ipz, t]                    #(jk, n_t)
    cov = (n_jk - 1) * np.cov(y_jk, rowvar=False, bias=True)    #(n_t, n_t): jackknife covariance
    if not pt2_correlated:
        cov = np.diag(np.diag(cov))

    fit = lsqfit.nonlinear_fit(data=(t, gv.gvar(y_jk.mean(axis=0), cov)), fcn=pt2_model, prior=pt2_prior(ipz), svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
    pt2_fit_result[pz] = fit
    if fit.stopping_criterion == 0:
        pt2_n_not_converged = pt2_n_not_converged + 1
    E_n = fit.p["E0"]
    for n in range(n_state):
        if n > 0:
            E_n = E_n + fit.p["dE"][n - 1]
        pt2_E_central[ipz, n] = E_n.mean
        pt2_E_lsqfit_err[ipz, n] = E_n.sdev
        pt2_c_central[ipz, n] = fit.p["c"][n].mean
        pt2_c_lsqfit_err[ipz, n] = fit.p["c"][n].sdev
    dof = fit.dof - n_state - 1     #lsqfit counts every prior as a data point; c_n and E0 have priors too wide to count
    pt2_chi2dof[ipz] = fit.chi2 / dof
    pt2_Q[ipz] = lsqfit.gammaQ(dof / 2, fit.chi2 / 2)

    for i in range(n_jk):
        fit_i = lsqfit.nonlinear_fit(data=(t, gv.gvar(y_jk[i], cov)), fcn=pt2_model, prior=pt2_prior(ipz), p0=fit.pmean, svdcut=pt2_svdcut, maxit=pt2_maxit, tol=pt2_tol, fitter=pt2_fitter)
        if fit_i.stopping_criterion == 0:
            pt2_n_not_converged = pt2_n_not_converged + 1
        E_i = fit_i.pmean["E0"]
        for n in range(n_state):
            if n > 0:
                E_i = E_i + fit_i.pmean["dE"][n - 1]
            pt2_E_jk[i, ipz, n] = E_i
            pt2_c_jk[i, ipz, n] = fit_i.pmean["c"][n]
gv.restore_gvar()

#---- print the result: jk = jackknife mean and error, lsqfit = the fit of the mean, its error sees the priors
print(f"{hadron_tag}, {frame}, {pt2_q_tag}, {pt2_fit_half}, {n_state} states, rest mass {m_rest:.4f}")
print(f"{'pz':>3s} {'t':>5s} {'E0 jk':>11s} {'E0 lsqfit':>11s} {'center':>8s} {'dE1 jk':>10s} {'dE1 lsqfit':>11s} {'c1/c0 jk':>10s} {'chi2/dof':>9s} {'Q':>6s}")
for ipz, pz in enumerate(pz_list):
    first, last = pt2_fit_window[pz]
    mean, err = jk(pt2_E_jk[:, ipz, 0])
    E0_text = str(gv.gvar(mean, err))
    E0_lsqfit_text = str(gv.gvar(pt2_E_central[ipz, 0], pt2_E_lsqfit_err[ipz, 0]))
    if n_state > 1:
        mean, err = jk(pt2_E_jk[:, ipz, 1] - pt2_E_jk[:, ipz, 0])
        dE1_text = str(gv.gvar(mean, err))
        dE1_lsqfit_text = str(pt2_fit_result[pz].p["dE"][0])
        mean, err = jk(pt2_c_jk[:, ipz, 1] / pt2_c_jk[:, ipz, 0])
        c_ratio_text = str(gv.gvar(mean, err))
    else:
        dE1_text = "-"
        dE1_lsqfit_text = "-"
        c_ratio_text = "-"
    print(f"{pz:3d} {first:2d}-{last:2d} {E0_text:>11s} {E0_lsqfit_text:>11s} {E0_center[ipz]:8.4f} {dE1_text:>10s} {dE1_lsqfit_text:>11s} {c_ratio_text:>10s} {pt2_chi2dof[ipz]:9.2f} {pt2_Q[ipz]:6.3f}")
print(f"fits that stopped at pt2_maxit before converging: {pt2_n_not_converged} of {n_pz * (n_jk + 1)}")

#---- save: the file states the fitted data and every setting
pt2_range_tag = "_".join(f"{pt2_fit_window[pz][0]}-{pt2_fit_window[pz][1]}n{n_state}" for pz in pz_list)     #first t - last t, number of states, one per pz
if pt2_fit_label == "":
    pt2_label_text = ""
else:
    pt2_label_text = "_" + pt2_fit_label
pt2_fit_file = os.path.join(save_dir, "2pt_fit_data", pt2_subfolder, f"2pt_fit_{frame}_{pt2_q_tag}_{gevp_tag}_t{pt2_range_tag}_{pt2_fit_half}_ncfg{ncfg}{pt2_label_text}_{hadron_tag}.h5")
if os.path.exists(pt2_fit_file):
    print("replacing", pt2_fit_file)
os.makedirs(os.path.dirname(pt2_fit_file), exist_ok=True)
with h5py.File(pt2_fit_file, "w") as f:
    #results
    f["fit_data_jk"] = pt2_fit_jk                           #(jk, pz, tsep): the data, for the plots
    f["E_jk"] = pt2_E_jk                                    #(jk, pz, state)
    f["c_jk"] = pt2_c_jk                                    #(jk, pz, state)
    f["E_central"] = pt2_E_central                          #(pz, state)
    f["E_lsqfit_err"] = pt2_E_lsqfit_err                    #(pz, state)
    f["c_central"] = pt2_c_central                          #(pz, state)
    f["c_lsqfit_err"] = pt2_c_lsqfit_err                    #(pz, state)
    f["chi2dof"] = pt2_chi2dof                              #(pz,)
    f["Q"] = pt2_Q                                          #(pz,)
    f["n_not_converged"] = pt2_n_not_converged
    #settings
    f["q_average_or_not"] = q_average_or_not
    f["single_q_data"] = single_q_data
    f["q_tag"] = pt2_q_tag
    f["subfolder"] = pt2_subfolder
    f["q_list"] = [q_list[i] for i in iq_use]               #(n_q, 3): the q that went in
    f["fit_half"] = pt2_fit_half
    f["fit_window"] = [pt2_fit_window[pz] for pz in pz_list]    #(pz, 2): first t, last t
    f["fit_range_tag"] = pt2_range_tag
    f["n_state"] = n_state
    f["E0_center_kind"] = pt2_E0_center
    f["E0_prior_center"] = E0_center                        #(pz,)
    f["E0_prior_width"] = pt2_E0_width
    f["m_rest"] = m_rest
    f["m_rest_from_meff"] = pt2_m_rest is None
    f["meff_pz"] = pt2_meff_pz
    f["meff_t"] = pt2_meff_t
    f["dE_prior"] = pt2_dE_prior                            #(center, width)
    f["c_prior"] = pt2_c_prior                              #(center, width)
    f["backward_wave"] = pt2_backward_wave
    f["correlated"] = pt2_correlated
    f["svdcut"] = pt2_svdcut
    f["maxit"] = pt2_maxit
    f["tol"] = pt2_tol
    f["fitter"] = pt2_fitter
    f["fit_label"] = pt2_fit_label
    #what the data are
    f["frame"] = frame
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["pf_list"] = pf_saved[iq_use[0]]                      #(pz, 3)
    f["Ls"] = Ls
    f["Lt"] = Lt
    save_gevp_info(f, op_basis)
print("saved", pt2_fit_file)

In [ ]:
#======== plot the saved 2pt fit: effective mass, one figure per pz, saved in meff_plots/{forward, sym_q..., sym_averaged_over_q} ========
#points: log[C(t) / C(t + 1)] at t + 1/2, filled inside the fit window, open outside;  blue band: the same of the fit curve of every jackknife sample
#gray band: E0;  dashed line: the center of the E0 prior
pt2_plot_file = pt2_fit_file        #the file cell 2 just saved;  in a new kernel paste here the path that cell 2 printed, as a string
meff_plot_t = np.arange(2, 19)      #the t of log[C(t) / C(t + 1)]
meff_plot_err_max = 0.1             #leave out points with a larger error, so noisy t do not set the y range
meff_color = "#2a78d6"

#---- read the saved fit
with h5py.File(pt2_plot_file, "r") as f:
    data_jk = f["fit_data_jk"][()]                      #(jk, pz, tsep)
    E_jk = f["E_jk"][()]                                #(jk, pz, state)
    c_jk = f["c_jk"][()]                                #(jk, pz, state)
    E_central = f["E_central"][()]                      #(pz, state)
    E_lsqfit_err = f["E_lsqfit_err"][()]                #(pz, state)
    chi2dof = f["chi2dof"][()]                          #(pz,)
    Q = f["Q"][()]                                      #(pz,)
    window = f["fit_window"][()]                        #(pz, 2)
    E0_prior_center = f["E0_prior_center"][()]          #(pz,)
    plot_pz_list = f["pz_list"][()]
    plot_n_state = f["n_state"][()]
    plot_Lt = f["Lt"][()]
    if f["backward_wave"][()]:
        backward_term = 1
    else:
        backward_term = 0
    plot_half = f["fit_half"].asstr()[()]
    plot_q_tag = f["q_tag"].asstr()[()]
    plot_subfolder = f["subfolder"].asstr()[()]
    plot_frame = f["frame"].asstr()[()]
    plot_hadron = f["hadron_tag"].asstr()[()]
    plot_ncfg = f["ncfg"][()]
    plot_gevp_tag = f["gevp_tag"].asstr()[()]
    plot_label = f["fit_label"].asstr()[()]
    plot_t0 = f["t0"][()]
    plot_t_ref = f["t_ref"][()]
    plot_basis = list(f["gevp_basis"].asstr()[()])

if plot_label == "":
    plot_label_text = ""
else:
    plot_label_text = "_" + plot_label
meff_plot_dir = os.path.join(save_dir, "meff_plots", plot_subfolder)
os.makedirs(meff_plot_dir, exist_ok=True)

#---- one figure per pz
for ipz, pz in enumerate(plot_pz_list):
    first, last = window[ipz]
    C_jk = data_jk[:, ipz]                              #(jk, tsep)
    with np.errstate(invalid="ignore"):                 #a negative C at a noisy t gives NaN, those points are dropped below
        m, e = jk(np.log(C_jk[:, meff_plot_t] / C_jk[:, meff_plot_t + 1]))
    m = np.where(e < meff_plot_err_max, m, np.nan)
    in_fit = (meff_plot_t >= first) & (meff_plot_t < last)      #both C(t) and C(t + 1) are in the fit

    #the band: the model at t and t + 1 for every jackknife sample
    t_band = np.linspace(first, last - 1, 60)           #(n_t,)
    E_n = E_jk[:, ipz, :, None]                         #(jk, state, 1)
    c_n = c_jk[:, ipz, :, None]                         #(jk, state, 1)
    C_t = (c_n * (np.exp(-E_n * t_band) + backward_term * np.exp(-E_n * (plot_Lt - t_band)))).sum(axis=1)                   #(jk, n_t)
    C_t1 = (c_n * (np.exp(-E_n * (t_band + 1)) + backward_term * np.exp(-E_n * (plot_Lt - t_band - 1)))).sum(axis=1)        #(jk, n_t)
    band_mean, band_err = jk(np.log(C_t / C_t1))

    mean, err = jk(E_jk[:, ipz, 0])
    E0 = gv.gvar(mean, err)
    E0_lsqfit = gv.gvar(E_central[ipz, 0], E_lsqfit_err[ipz, 0])
    if plot_n_state > 1:
        mean, err = jk(E_jk[:, ipz, 1] - E_jk[:, ipz, 0])
        dE1_text = f"dE1 = {gv.gvar(mean, err)}, "
    else:
        dE1_text = ""

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.axhspan(E0.mean - E0.sdev, E0.mean + E0.sdev, color="#e7e7e5", lw=0, zorder=0, label="E0")
    ax.axhline(E0_prior_center[ipz], color="#52514e", lw=1, ls="--", zorder=1, label="E0 prior center")
    ax.fill_between(t_band + 0.5, band_mean - band_err, band_mean + band_err, color=meff_color, alpha=0.25, lw=0, zorder=1, label="fit")
    ax.errorbar(meff_plot_t[in_fit] + 0.5, m[in_fit], e[in_fit], color=meff_color, marker="o", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="in the fit")
    ax.errorbar(meff_plot_t[~in_fit] + 0.5, m[~in_fit], e[~in_fit], color=meff_color, marker="o", mfc="white", ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label="not in the fit")
    #y range from the points in the fit, so the small t do not squash the plateau
    y_low = np.nanmin(np.append(m[in_fit] - e[in_fit], [E0.mean - E0.sdev, E0_prior_center[ipz]]))
    y_high = np.nanmax(np.append(m[in_fit] + e[in_fit], [E0.mean + E0.sdev, E0_prior_center[ipz]]))
    ax.set_ylim(y_low - 0.1 * (y_high - y_low), y_high + 0.1 * (y_high - y_low))
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("t")
    ax.set_ylabel(r"$\log[C(t - 1/2) / C(t + 1/2)]$")
    ax.set_title(f"{plot_hadron}, {plot_frame}, {plot_q_tag}, {plot_half}, pz = {pz}, {plot_ncfg} cfg\n"
                 f"{' '.join(plot_basis)}, t0 {plot_t0}, t_ref {plot_t_ref}\n"
                 f"t {first}-{last}, {plot_n_state} states: E0 = {E0} (lsqfit {E0_lsqfit}), {dE1_text}chi2/dof {chi2dof[ipz]:.2f}, Q {Q[ipz]:.3f}", fontsize=9)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(meff_plot_dir, f"meff_{plot_frame}_{plot_q_tag}_{plot_gevp_tag}_t{first}-{last}n{plot_n_state}_{plot_half}_ncfg{plot_ncfg}{plot_label_text}_{plot_hadron}_pz{pz}.png"), dpi=150)
    plt.close(fig)
print("saved", len(plot_pz_list), "plots in", meff_plot_dir)

In [ ]:
#======== 3pt, method 1: one file per q ========
pt3_q_list = q_list         #the q to make;  q_list = all of them, or for example [(2,0,0)]
tau_list = np.arange(20)
half_sign = {"forward": +1, "backward": -1}     #operator at t_src + tau (forward), t_src - tau (backward)

print("GEVP basis", " ".join(op_labels), "| t0", t0, "| t_ref", t_ref)
assert phase_attached_to in ["FF", "2pt"], f"phase_attached_to = {phase_attached_to}: use FF or 2pt"
assert os.path.exists(gevp_file), f"no GEVP file for {gevp_tag}: run cell 1 first"

#---- the saved 2pt and v0 of this basis
C2_saved = {}
v0_saved = {}
with h5py.File(gevp_file, "r") as f:
    for half in half_sign:
        C2_saved[half] = f[f"C2_{half}"][()]    #(cfg, q, pz, tsep)
        v0_saved[half] = f[f"v0_{half}"][()]    #(q, pz, op)

for q_i in pt3_q_list:
    print("3pt at q =", q_i)
    iq = q_list.index(q_i)
    iq_minus = q_list.index((-q_i[0], -q_i[1], -q_i[2]))

    #---- source positions, and C3input = v^dagger C v with the saved v0
    C3input_file = pt2_template.format(frame=frame, qx=q_i[0], qy=q_i[1], qz=q_i[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag)
    with h5py.File(C3input_file, "r") as f:
        cfg_list = f["cfg_list"][()]
        pz_list = f["pz_list"][()]
        t_src_list = f["t_src_list"][()]        #(cfg, t_src): the slice of each t_src index, not sorted, shifts per cfg
        x_src = f["x_src_list"][()]             #(cfg, 4): source positions, they shift per cfg
        y_src = f["y_src_list"][()]             #(cfg, 4)
        z_src = f["z_src_list"][()]             #(cfg, 8)
    raw_forward, raw_backward = read_pt2(q_i, "C3input")
    C3input = {}                                #C3input[half]: (cfg, t_src, pz, tsep), c_sign already in
    C3input["forward"] = apply_v0(op_matrix(raw_forward, op_basis), v0_saved["forward"][iq])
    C3input["backward"] = apply_v0(op_matrix(raw_backward, op_basis), v0_saved["backward"][iq])

    #mean of exp(-i 2pi/Ls q.x_src) over the 128 source positions, one number per cfg: 1 at q = 0, ~1e-16 at the four sym q
    phase_mean = (np.exp(-2j * np.pi * q_i[0] * x_src / Ls).mean(axis=1)
                  * np.exp(-2j * np.pi * q_i[1] * y_src / Ls).mean(axis=1)
                  * np.exp(-2j * np.pi * q_i[2] * z_src / Ls).mean(axis=1))

    #---- the gluon operator: FF (cfg, munu, rhosig, hyp, w, t), munu order XY, YZ, XZ, XT, YT, ZT
    FF_file = os.path.join(pt2_dir, f"FF_hyp0-8_w0-9_q{q_i[0]}_{q_i[1]}_{q_i[2]}_ncfg800.h5")
    with h5py.File(FF_file, "r") as f:
        assert tuple(f["q"][()]) == q_i
        FF_cfg = f["cfg_list"][()]
        idx = np.searchsorted(FF_cfg, cfg_list)     #the 2pt cfgs are not the first 250 FF cfgs
        assert np.array_equal(FF_cfg[idx], cfg_list)
        ihyp = list(f["hyp_list"][()]).index(hyp)
        w_list = f["w_list"][()]
        TXTX = f["FF"][idx, 3, 3, ihyp]
        TYTY = f["FF"][idx, 4, 4, ihyp]
        XYXY = f["FF"][idx, 0, 0, ihyp]
    O = TXTX + TYTY - 2 * XYXY                      #(cfg, w, t)

    #---- connected minus vacuum, for every jackknife sample
    n_cfg, n_tsrc = t_src_list.shape
    i_cfg = np.arange(n_cfg)[:, None, None]
    pt3_jk = {}
    C2pf_jk = {}
    C2pi_jk = {}
    for half in half_sign:
        C2pf_cfg = C2_saved[half][:, iq]            #(cfg, pz, tsep): plain 2pt at p_f = q/2
        C2pi_cfg = C2_saved[half][:, iq_minus]      #(cfg, pz, tsep): plain 2pt at p_i = -q/2, C2pi(q) = C2pf(-q)
        t_op = (t_src_list[:, :, None] + half_sign[half] * tau_list) % Lt       #(cfg, t_src, tau)
        O_tau = O[i_cfg, :, t_op]                   #(cfg, t_src, tau, w): numpy puts w last here
        connected = np.einsum("cspT,cstw->cwpTt", C3input[half], O_tau, optimize=True) / n_tsrc     #(cfg, w, pz, tsep, tau)
        if phase_attached_to == "FF":
            pt2_avg = C2pf_cfg                      #(cfg, pz, tsep), plain, no source phase
            op_avg = O_tau.mean(axis=1) * phase_mean[:, None, None]     #(cfg, tau, w), the source phase is here
        else:
            pt2_avg = C3input[half].mean(axis=1)    #(cfg, pz, tsep), the source phase is here
            op_avg = O_tau.mean(axis=1)             #(cfg, tau, w), no source phase
        pt3_jk[half] = jackknife(connected) - np.einsum("cpT,ctw->cwpTt", jackknife(pt2_avg), jackknife(op_avg), optimize=True)    #(jk, w, pz, tsep, tau)
        C2pf_jk[half] = jackknife(C2pf_cfg).real    #(jk, pz, tsep)
        C2pi_jk[half] = jackknife(C2pi_cfg).real    #(jk, pz, tsep)

    #check: C3/C2 at tsep 8, tau 4, w 0
    for half in half_sign:
        m, e = jk(pt3_jk[half][:, 0, :, 8, 4].real / C2pf_jk[half][:, :, 8])
        print(half, " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

    #---- save the 3pt, with the jackknife 2pt that goes with it
    os.makedirs(os.path.dirname(pt3_path(q_i)), exist_ok=True)
    with h5py.File(pt3_path(q_i), "w") as f:
        for half in half_sign:
            f[f"pt3_{half}"] = pt3_jk[half]         #(jk, w, pz, tsep, tau), complex
            f[f"C2pf_{half}"] = C2pf_jk[half]       #(jk, pz, tsep)
            f[f"C2pi_{half}"] = C2pi_jk[half]       #(jk, pz, tsep)
        f["q"] = q_i
        f["cfg_list"] = cfg_list
        f["pz_list"] = pz_list
        f["w_list"] = w_list
        f["tsep_list"] = np.arange(pt3_jk["forward"].shape[3])
        f["tau_list"] = tau_list
        f["hyp"] = hyp
        save_gevp_info(f, op_basis)
        f["phase_attached_to"] = phase_attached_to
    print("saved", pt3_path(q_i))

In [ ]:
#======== ratio, and its plots: one figure per w and pz ========
#R(tsep, tau) = C3(tsep, tau) / C2pf(tsep) * sqrt[ C2pi(tsep - tau) C2pf(tau) C2pf(tsep) / (C2pf(tsep - tau) C2pi(tau) C2pi(tsep)) ]
#the ground state gives M / (2 sqrt(E_f E_i)) at every tsep, tau;  at q = 0 the square root is 1
ratio_q_average = True                      #sym only: True = average the 3pt and 2pt over the four q, then make the ratio;  False = the q of cell 0
plot_half = "fb"                            #"forward", "backward" or "fb"
plot_tsep_list = [4,5,6,7,8,9,10,11]        #up to 8 values
plot_tau_skip = 1                           #leave out tau < plot_tau_skip and tau > tsep - plot_tau_skip
plot_err_max = 0.3                          #leave out points with a larger error, so noisy tsep do not set the y range

print("GEVP basis", " ".join(op_labels), "| t0", t0, "| t_ref", t_ref)
half_list = ["forward", "backward"]

#---- which q go in, and the label of this ratio
if frame == "forward":
    ratio_q_list = [(0,0,0)]
    ratio_q_tag = "q0_0_0"
    ratio_subfolder = "forward"
elif ratio_q_average:
    ratio_q_list = q_list
    ratio_q_tag = "averaged_over_q"
    ratio_subfolder = "sym_averaged_over_q"
else:
    ratio_q_list = [q]
    ratio_q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
    ratio_subfolder = f"sym_{ratio_q_tag}"

#---- read the 3pt files and average over q
pt3_use = {"forward": 0, "backward": 0}     #pt3_use[half]: (jk, w, pz, tsep, tau)
c2f_jk = {"forward": 0, "backward": 0}      #c2f_jk[half]: (jk, pz, tsep)
c2i_jk = {"forward": 0, "backward": 0}      #c2i_jk[half]: (jk, pz, tsep)
for q_i in ratio_q_list:
    assert os.path.exists(pt3_path(q_i)), f"no 3pt file at q = {q_i} for {gevp_tag}: run cell 4 first"
    with h5py.File(pt3_path(q_i), "r") as f:
        if q_i == ratio_q_list[0]:
            cfg_list = f["cfg_list"][()]
            pz_list = f["pz_list"][()]
            w_list = f["w_list"][()]
            tsep_list = f["tsep_list"][()]
            tau_list = f["tau_list"][()]
        assert np.array_equal(f["cfg_list"][()], cfg_list), f"q = {q_i}: other cfgs, the jackknife samples would not match"
        for half in half_list:
            pt3_use[half] = pt3_use[half] + f[f"pt3_{half}"][()] / len(ratio_q_list)
            c2f_jk[half] = c2f_jk[half] + f[f"C2pf_{half}"][()] / len(ratio_q_list)
            c2i_jk[half] = c2i_jk[half] + f[f"C2pi_{half}"][()] / len(ratio_q_list)

#---- the ratio
tsep_grid, tau_grid = np.meshgrid(tsep_list, tau_list, indexing="ij")           #(tsep, tau)
in_window = tau_grid <= tsep_grid
rest_grid = np.where(in_window, tsep_grid - tau_grid, 0)                        #tsep - tau, the 0 is only used where R is set to NaN


def build_ratio(pt3, c2f, c2i):
    #pt3 (jk, w, pz, tsep, tau), c2f and c2i real (jk, pz, tsep) -> R (jk, w, pz, tsep, tau), complex, NaN for tau > tsep
    root = np.sqrt(c2i[:, :, rest_grid] * c2f[:, :, tau_grid] * c2f[:, :, tsep_grid]
                   / (c2f[:, :, rest_grid] * c2i[:, :, tau_grid] * c2i[:, :, tsep_grid]))    #(jk, pz, tsep, tau)
    return np.where(in_window, pt3 * (root / c2f[:, :, tsep_grid])[:, None], np.nan)


ratio_jk = {}
for half in half_list:
    ratio_jk[half] = build_ratio(pt3_use[half], c2f_jk[half], c2i_jk[half])
#forward and backward together: average the jackknife 3pt and 2pt first, then the ratio
ratio_jk["fb"] = build_ratio(0.5 * (pt3_use["forward"] + pt3_use["backward"]),
                             0.5 * (c2f_jk["forward"] + c2f_jk["backward"]),
                             0.5 * (c2i_jk["forward"] + c2i_jk["backward"]))

#check: Re R at tsep 8, tau 4, w 0
for half in ratio_jk:
    m, e = jk(ratio_jk[half][:, 0, :, 8, 4].real)
    print(f"{half:8s}", " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

#---- save the ratio
ratio_file = os.path.join(save_dir, "ratio_data", ratio_subfolder, f"ratio_{frame}_{ratio_q_tag}_{gevp_tag}_phase{phase_attached_to}_hyp{hyp}_ncfg{ncfg}_{hadron_tag}.h5")
os.makedirs(os.path.dirname(ratio_file), exist_ok=True)
with h5py.File(ratio_file, "w") as f:
    for half in ratio_jk:
        f[f"ratio_{half}"] = ratio_jk[half]         #(jk, w, pz, tsep, tau), complex, NaN for tau > tsep;  half = forward, backward, fb
    f["q_list"] = ratio_q_list                      #(n_q, 3): the q that went in
    f["q_tag"] = ratio_q_tag
    f["subfolder"] = ratio_subfolder
    f["frame"] = frame
    f["hadron_tag"] = hadron_tag
    f["ncfg"] = ncfg
    f["cfg_list"] = cfg_list
    f["pz_list"] = pz_list
    f["w_list"] = w_list
    f["tsep_list"] = tsep_list
    f["tau_list"] = tau_list
    f["hyp"] = hyp
    save_gevp_info(f, op_basis)
    f["phase_attached_to"] = phase_attached_to
print("saved", ratio_file)

#---- plot: saved in ratio_plots/{forward, sym_q..., sym_averaged_over_q}/pz{pz}
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
ratio_plot_dir = os.path.join(save_dir, "ratio_plots", ratio_subfolder)
for iw, w in enumerate(w_list):
    for ipz, pz in enumerate(pz_list):
        pz_dir = os.path.join(ratio_plot_dir, f"pz{pz}")
        os.makedirs(pz_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        for i, tsep in enumerate(plot_tsep_list):
            tau = np.arange(plot_tau_skip, tsep - plot_tau_skip + 1)
            m, e = jk(ratio_jk[plot_half][:, iw, ipz, tsep, tau].real)
            m = np.where(e < plot_err_max, m, np.nan)                           #NaN leaves a gap in the line instead of joining across the dropped points
            x = tau - tsep / 2 + 0.06 * (i - (len(plot_tsep_list) - 1) / 2)     #small shift so the error bars do not sit on top of each other
            ax.errorbar(x, m, e, color=colors[i], marker=markers[i], ms=5, lw=0.8, elinewidth=1, capsize=2, label=f"tsep = {tsep}")
        ax.axhline(0, color="#898781", lw=0.8)
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$")
        ax.set_ylabel("Re R")
        ax.set_title(f"{hadron_tag}, {frame}, {ratio_q_tag}, {plot_half}, phase on {phase_attached_to}, hyp {hyp}, w = {w}, pz = {pz}, {ncfg} cfg", fontsize=10)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(pz_dir, f"ratio_{frame}_{ratio_q_tag}_{gevp_tag}_phase{phase_attached_to}_{hadron_tag}_{plot_half}_hyp{hyp}_w{w}_pz{pz}.png"), dpi=150)
        plt.close(fig)
print("saved", len(w_list) * len(pz_list), "plots in", ratio_plot_dir)

In [ ]:
#======== ratio fit, and its plots: every fit parameter is here ========
#the model: R = M00 + Ai (e^{-dE tau} + e^{-dE (tsep - tau)}) + Afi e^{-dE tsep},  one fit per pz and w, each with its own gap
fit_ratio_file = ratio_file         #the file cell 5 just saved;  in a new kernel paste here the path that cell 5 printed, as a string
fit_half = "fb"                     #"forward", "backward" or "fb"
fit_window = {0: (6, 12, 3), 1: (6, 12, 3), 2: (6, 12, 3), 3: (4, 10, 2), 4: (4, 9, 2), 5: (4, 9, 2), 6: (4, 8, 2)}    #pz: (first tsep, last tsep, tau_skip);  tau = tau_skip .. tsep - tau_skip
fit_dE_prior = (0.5, 0.5)           #(center, width) of the gap at every pz and w;  the fit varies log(dE), so dE stays positive
fit_M00_prior = (0.0, 1e4)          #(center, width)
fit_Ai_prior = (0.0, 1e4)           #(center, width)
fit_Afi_prior = (0.0, 1e4)          #(center, width)
fit_correlated = True               #True = full covariance;  False = only its diagonal
fit_svdcut = 1e-7
fit_maxit = 20000
fit_tol = 1e-8                      #stopping tolerance of the fit
fit_fitter = "scipy_least_squares"  #the minimizer
fit_label = ""                      #extra text in the file name;  set it when you change a prior, svdcut, ..., or the old file is replaced

#---- read the ratio;  the labels of this fit come from the file
assert fit_half in ["forward", "backward", "fb"], f"fit_half = {fit_half}: use forward, backward or fb"
with h5py.File(fit_ratio_file, "r") as f:
    fit_data_jk = f[f"ratio_{fit_half}"][()]        #(jk, w, pz, tsep, tau), complex
    fit_w_list = f["w_list"][()]
    fit_pz_list = f["pz_list"][()]
    fit_q_tag = f["q_tag"].asstr()[()]
    fit_subfolder = f["subfolder"].asstr()[()]
    fit_frame = f["frame"].asstr()[()]
    fit_hadron = f["hadron_tag"].asstr()[()]
    fit_ncfg = f["ncfg"][()]
    fit_hyp = f["hyp"][()]
    fit_phase = f["phase_attached_to"].asstr()[()]
    fit_gevp_tag = f["gevp_tag"].asstr()[()]
print("ratio file", fit_ratio_file)
for pz in fit_pz_list:
    assert pz in fit_window, f"pz = {pz} is missing from fit_window"


#---- the model and the prior
def ratio_model(x, p):
    #no denominator;  forward and sym: p_i and p_f have the same energies and overlaps, so one gap and Af = Ai
    tsep = x["tsep"]
    tau = x["tau"]
    return p["M00"] + p["Ai"] * (gv.exp(-p["dE"] * tau) + gv.exp(-p["dE"] * (tsep - tau))) + p["Afi"] * gv.exp(-p["dE"] * tsep)


def ratio_prior():
    prior = gv.BufferDict()
    prior["M00"] = gv.gvar(fit_M00_prior[0], fit_M00_prior[1])
    prior["Ai"] = gv.gvar(fit_Ai_prior[0], fit_Ai_prior[1])
    prior["Afi"] = gv.gvar(fit_Afi_prior[0], fit_Afi_prior[1])
    prior["log(dE)"] = gv.log(gv.gvar(fit_dE_prior[0], fit_dE_prior[1]))    #read the gap as fit.p["dE"], fit.psdev["dE"] is not its error
    return prior


#---- one fit per pz and w: first the jackknife mean, then every jackknife sample with the same covariance and prior, started from the fit of the mean
n_jk = fit_data_jk.shape[0]
n_w = len(fit_w_list)
n_pz = len(fit_pz_list)
fit_param_names = ["M00", "Ai", "Afi", "dE"]
param_jk = {}           #param_jk[name]: (jk, w, pz)
param_central = {}      #param_central[name]: (w, pz), the fit of the jackknife mean
param_lsqfit_err = {}   #param_lsqfit_err[name]: (w, pz), its lsqfit error, which sees the dE prior
for name in fit_param_names:
    param_jk[name] = np.zeros((n_jk, n_w, n_pz))
    param_central[name] = np.zeros((n_w, n_pz))
    param_lsqfit_err[name] = np.zeros((n_w, n_pz))
fit_chi2dof = np.zeros((n_w, n_pz))     #(w, pz)
fit_Q = np.zeros((n_w, n_pz))           #(w, pz)
fit_points = {}                         #fit_points[pz]: the tsep and tau of every point
fit_n_not_converged = 0                 #how many fits stopped at fit_maxit before converging;  lsqfit gives no warning for this
gv.switch_gvar()        #the fits keep their covariances apart and gv.restore_gvar() below frees them:  without this, every rerun of this cell takes more memory and time
for ipz, pz in enumerate(fit_pz_list):
    first, last, tau_skip = fit_window[pz]
    tsep_points = []
    tau_points = []
    for tsep in range(first, last + 1):
        for tau in range(tau_skip, tsep - tau_skip + 1):
            tsep_points.append(tsep)
            tau_points.append(tau)
    assert len(tsep_points) > 3, f"pz = {pz}: {len(tsep_points)} points for 3 free parameters, make the window larger"
    x = {"tsep": np.array(tsep_points), "tau": np.array(tau_points)}    #both (n_point,)
    fit_points[pz] = x
    for iw, w in enumerate(fit_w_list):
        y_jk = fit_data_jk[:, iw, ipz, x["tsep"], x["tau"]].real        #(jk, n_point): Re R
        assert not np.isnan(y_jk).any(), f"pz = {pz}: NaN in the ratio at the fit points, lower the last tsep in fit_window"
        cov = (n_jk - 1) * np.cov(y_jk, rowvar=False, bias=True)        #(n_point, n_point): jackknife covariance
        if not fit_correlated:
            cov = np.diag(np.diag(cov))

        fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk.mean(axis=0), cov)), fcn=ratio_model, prior=ratio_prior(), svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
        if fit.stopping_criterion == 0:
            fit_n_not_converged = fit_n_not_converged + 1
        dof = fit.dof - 3       #lsqfit counts every prior as a data point; M00, Ai, Afi have priors too wide to count
        fit_chi2dof[iw, ipz] = fit.chi2 / dof
        fit_Q[iw, ipz] = lsqfit.gammaQ(dof / 2, fit.chi2 / 2)
        for name in fit_param_names:
            param_central[name][iw, ipz] = fit.p[name].mean
            param_lsqfit_err[name][iw, ipz] = fit.p[name].sdev

        for i in range(n_jk):
            fit_i = lsqfit.nonlinear_fit(data=(x, gv.gvar(y_jk[i], cov)), fcn=ratio_model, prior=ratio_prior(), p0=fit.pmean, svdcut=fit_svdcut, maxit=fit_maxit, tol=fit_tol, fitter=fit_fitter)
            if fit_i.stopping_criterion == 0:
                fit_n_not_converged = fit_n_not_converged + 1
            for name in fit_param_names:
                param_jk[name][i, iw, ipz] = fit_i.pmean[name]
gv.restore_gvar()

#---- print the result: jk = jackknife mean and error, lsqfit = the fit of the mean, its error sees the dE prior
print(f"{fit_hadron}, {fit_frame}, {fit_q_tag}, {fit_half}, hyp {fit_hyp}")
print(" pz   w  points       M00 jk        dE jk   M00 lsqfit   chi2/dof      Q")
for ipz, pz in enumerate(fit_pz_list):
    for iw, w in enumerate(fit_w_list):
        mean, err = jk(param_jk["M00"][:, iw, ipz])
        M00_text = str(gv.gvar(mean, err))
        mean, err = jk(param_jk["dE"][:, iw, ipz])
        dE_text = str(gv.gvar(mean, err))
        M00_lsqfit_text = str(gv.gvar(param_central["M00"][iw, ipz], param_lsqfit_err["M00"][iw, ipz]))
        print(f"{pz:3d} {w:3d} {len(fit_points[pz]['tsep']):7d} {M00_text:>12s} {dE_text:>12s} {M00_lsqfit_text:>12s} {fit_chi2dof[iw, ipz]:10.2f} {fit_Q[iw, ipz]:6.3f}")
print(f"fits that stopped at fit_maxit before converging: {fit_n_not_converged} of {n_pz * n_w * (n_jk + 1)}")

#---- save: the file states every setting
fit_range_tag = "_".join(f"{fit_window[pz][0]}-{fit_window[pz][1]}s{fit_window[pz][2]}" for pz in fit_pz_list)     #first tsep - last tsep, tau_skip, one per pz
if fit_label == "":
    fit_label_text = ""
else:
    fit_label_text = "_" + fit_label
fit_file = os.path.join(save_dir, "ratio_fit_data", fit_subfolder, f"ratio_fit_{fit_frame}_{fit_q_tag}_{fit_gevp_tag}_tsep{fit_range_tag}_phase{fit_phase}_ratio_{fit_half}_hyp{fit_hyp}_ncfg{fit_ncfg}{fit_label_text}_{fit_hadron}.h5")
if os.path.exists(fit_file):
    print("replacing", fit_file)
os.makedirs(os.path.dirname(fit_file), exist_ok=True)
with h5py.File(fit_file, "w") as f, h5py.File(fit_ratio_file, "r") as f_ratio:
    #results
    for name in fit_param_names:
        f[f"{name}_jk"] = param_jk[name]                    #(jk, w, pz)
        f[f"{name}_central"] = param_central[name]          #(w, pz)
        f[f"{name}_lsqfit_err"] = param_lsqfit_err[name]    #(w, pz)
    f["chi2dof"] = fit_chi2dof                              #(w, pz)
    f["Q"] = fit_Q                                          #(w, pz)
    f["n_not_converged"] = fit_n_not_converged
    #settings
    f["ratio_file"] = fit_ratio_file
    f["model"] = "ratio"
    f["fit_half"] = fit_half
    f["fit_window"] = [fit_window[pz] for pz in fit_pz_list]    #(pz, 3): first tsep, last tsep, tau_skip
    f["fit_range_tag"] = fit_range_tag
    f["dE_prior"] = fit_dE_prior                            #(center, width)
    f["M00_prior"] = fit_M00_prior                          #(center, width)
    f["Ai_prior"] = fit_Ai_prior                            #(center, width)
    f["Afi_prior"] = fit_Afi_prior                          #(center, width)
    f["correlated"] = fit_correlated
    f["svdcut"] = fit_svdcut
    f["maxit"] = fit_maxit
    f["tol"] = fit_tol
    f["fitter"] = fit_fitter
    f["fit_label"] = fit_label
    #what the data are, as stated in the ratio file
    for name in ["q_list", "q_tag", "subfolder", "frame", "hadron_tag", "ncfg", "cfg_list", "pz_list", "w_list", "hyp", "phase_attached_to",
                 "gevp_tag", "gevp_basis", "op_basis", "smear_names", "smear_list", "gamma_names", "t0", "t_ref"]:
        f_ratio.copy(name, f)
print("saved", fit_file)

#---- plot: saved in ratio_fit_plots/{forward, sym_q..., sym_averaged_over_q}/pz{pz}
#points: the fitted Re R;  colored bands: the fit curve of every jackknife sample, one per tsep;  gray band: M00
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
fit_plot_dir = os.path.join(save_dir, "ratio_fit_plots", fit_subfolder)
for ipz, pz in enumerate(fit_pz_list):
    pz_dir = os.path.join(fit_plot_dir, f"pz{pz}")
    os.makedirs(pz_dir, exist_ok=True)
    x = fit_points[pz]
    tsep_values = np.unique(x["tsep"])
    first, last, tau_skip = fit_window[pz]
    for iw, w in enumerate(fit_w_list):
        y_jk = fit_data_jk[:, iw, ipz, x["tsep"], x["tau"]].real        #(jk, n_point)
        p_jk = {}
        for name in fit_param_names:
            p_jk[name] = param_jk[name][:, iw, ipz, None]               #(jk, 1), so the model below gives (jk, n_tau)
        mean, err = jk(param_jk["M00"][:, iw, ipz])
        M00 = gv.gvar(mean, err)
        mean, err = jk(param_jk["dE"][:, iw, ipz])
        dE = gv.gvar(mean, err)
        M00_lsqfit = gv.gvar(param_central["M00"][iw, ipz], param_lsqfit_err["M00"][iw, ipz])
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.axhspan(M00.mean - M00.sdev, M00.mean + M00.sdev, color="#e7e7e5", lw=0, zorder=0, label="M00")
        for i, tsep in enumerate(tsep_values):
            on_tsep = x["tsep"] == tsep
            tau = x["tau"][on_tsep]
            shift = 0.06 * (i - (len(tsep_values) - 1) / 2)     #small shift so the error bars do not sit on top of each other
            #0.25 past the last points on each side, else the tsep with one point has no width
            tau_band = np.linspace(tau[0] - 0.25, tau[-1] + 0.25, 40)
            band_mean, band_err = jk(ratio_model({"tsep": tsep, "tau": tau_band}, p_jk))
            ax.fill_between(tau_band - tsep / 2 + shift, band_mean - band_err, band_mean + band_err, color=colors[i], alpha=0.25, lw=0, zorder=1)
            m, e = jk(y_jk[:, on_tsep])
            ax.errorbar(tau - tsep / 2 + shift, m, e, color=colors[i], marker=markers[i], ms=5, ls="none", elinewidth=1, capsize=2, zorder=2, label=f"tsep = {tsep}")
        ax.grid(color="#e1e0d9", lw=0.6)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel(r"$\tau - t_{\rm sep}/2$")
        ax.set_ylabel("Re R")
        ax.set_title(f"{fit_hadron}, {fit_frame}, {fit_q_tag}, {fit_half}, phase on {fit_phase}, hyp {fit_hyp}, w = {w}, pz = {pz}, {fit_ncfg} cfg\n"
                     f"M00 = {M00} (lsqfit {M00_lsqfit}), dE = {dE}, chi2/dof {fit_chi2dof[iw, ipz]:.2f}, Q {fit_Q[iw, ipz]:.3f}", fontsize=9)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
        fig.tight_layout()
        fig.savefig(os.path.join(pz_dir, f"ratio_fit_{fit_frame}_{fit_q_tag}_{fit_gevp_tag}_tsep{first}-{last}s{tau_skip}_phase{fit_phase}_{fit_hadron}_ratio_{fit_half}_hyp{fit_hyp}{fit_label_text}_w{w}_pz{pz}.png"), dpi=150)
        plt.close(fig)
print("saved", n_w * n_pz, "plots in", fit_plot_dir)

In [ ]:
#======== single and double ratios: the fits are looked up from the GEVP tag;  only the first cell has to be run before this one ========
import gvar as gv

#---- what to use
itd_basis = ["BG5", "BG5G4", "BG3G5", "UG5", "UG5G4", "UG3G5"]      #the GEVP operators, in the order of op_basis
itd_t0 = 1
itd_t_ref = 3
itd_average_over_q = True           #sym only: True = the fits of the q average, False = the fits at the q of the first cell
itd_ref = "pz0"                     #sym only, the reference at nu = 0: "pz0" = the P_z = 0 entry of this run (the same t), "p000" = p = (0,0,0) of the forward run (t = 0);  forward: always p = (0,0,0)
itd_w_ref = 0                       #double ratio: every matrix element is divided by its value at this w
itd_file_filter = {"ratio_fit": "", "2pt_fit": ""}      #only when a tag has several fits: text in the name of the one to use, e.g. "tsep6-12s3_6-12s3_6-12s3_4-10s2" or "hyp8"
itd_plot_err_max = 0.5              #leave out points with a larger error, so noisy points do not set the y range

itd_root = os.path.join(pt2_dir, "AI_assisted_matrix_element_analysis")
itd_gevp_tag = "_".join(itd_basis) + f"_t0{itd_t0}_tref{itd_t_ref}"
itd_q_tag = "averaged_over_q" if itd_average_over_q and frame == "sym" else f"q{q[0]}_{q[1]}_{q[2]}"
itd_subfolder = "forward" if frame == "forward" else f"sym_{itd_q_tag}"
assert itd_ref in ["pz0", "p000"], f"itd_ref = {itd_ref}: use pz0 or p000"
itd_ref_used = "p000" if frame == "forward" else itd_ref


#---- find and read the fits
def find_fit(kind, fit_frame, q_tag_i, subfolder_i):
    #the one saved fit with this GEVP tag;  kind = "ratio_fit" or "2pt_fit"
    folder = os.path.join(itd_root, f"{kind}_data", subfolder_i)
    files = glob.glob(os.path.join(folder, f"{kind}_{fit_frame}_{q_tag_i}_{itd_gevp_tag}_*_{hadron_tag}.h5"))
    files = [file for file in files if itd_file_filter[kind] in os.path.basename(file)]
    available = "\n".join(sorted(os.listdir(folder))) if os.path.isdir(folder) else f"no folder {folder}"
    assert len(files) == 1, f"{len(files)} {kind} files for tag {itd_gevp_tag}, {hadron_tag}, {subfolder_i};  use itd_file_filter if there are several.  In the folder:\n{available}"
    return files[0]


def read_fits(fit_frame, q_tag_i, subfolder_i):
    #M00 (jk, w, pz) of the ratio fit and the central E0 (pz,) of the 2pt fit with the same GEVP tag
    fits = {"ratio_fit_file": find_fit("ratio_fit", fit_frame, q_tag_i, subfolder_i), "pt2_fit_file": find_fit("2pt_fit", fit_frame, q_tag_i, subfolder_i)}
    with h5py.File(fits["ratio_fit_file"], "r") as f, h5py.File(fits["pt2_fit_file"], "r") as g:
        for fit in [f, g]:
            assert fit["gevp_tag"].asstr()[()] == itd_gevp_tag, f"{fit.filename}: the file states another GEVP tag"
        assert np.array_equal(f["cfg_list"][()], g["cfg_list"][()]) and np.array_equal(f["pz_list"][()], g["pz_list"][()]), "the ratio fit and the 2pt fit have other cfgs or pz"
        assert f["fit_half"][()] == g["fit_half"][()], "the ratio fit and the 2pt fit use another half"
        fits["M00_jk"] = f["M00_jk"][()]
        fits["E_central"] = g["E_central"][:, 0]
        fits["w_list"], fits["pz_list"], fits["cfg_list"] = f["w_list"][()], f["pz_list"][()], f["cfg_list"][()]
        fits["ratio_fit_range"], fits["pt2_fit_range"] = f["fit_range_tag"].asstr()[()], g["fit_range_tag"].asstr()[()]
        fits["Ls"] = g["Ls"][()]
    return fits


run_fits = read_fits(frame, itd_q_tag, itd_subfolder)
M_jk, E_central, w_values, pz_values = run_fits["M00_jk"], run_fits["E_central"], run_fits["w_list"], run_fits["pz_list"]
if itd_ref_used == "p000" and frame == "sym":
    ref_fits = read_fits("forward", "q0_0_0", "forward")
    assert np.array_equal(ref_fits["cfg_list"], run_fits["cfg_list"]) and np.array_equal(ref_fits["w_list"], w_values), "the forward run has other cfgs or w"
else:
    ref_fits = run_fits                 #forward: pz 0 is p = (0,0,0);  sym with pz0: pz 0 is the P_z = 0 entry
ipz_ref = list(ref_fits["pz_list"]).index(0)
M_ref_jk = ref_fits["M00_jk"][:, :, ipz_ref, None]      #(jk, w, 1)
E_ref = ref_fits["E_central"][ipz_ref]


#---- the ratios, sample by sample;  E_ref / E is the central value for every sample
iw_ref = list(w_values).index(itd_w_ref)
nu = 2 * np.pi / run_fits["Ls"] * np.outer(w_values, pz_values)                                         #(w, pz)
single_jk = M_jk / M_ref_jk * (E_ref / E_central)                                                       #(jk, w, pz)
double_jk = (M_jk / M_jk[:, iw_ref:iw_ref + 1]) / (M_ref_jk / M_ref_jk[:, iw_ref:iw_ref + 1])           #(jk, w, pz)
single_mean, single_err = jk(single_jk)
double_mean, double_err = jk(double_jk)
print(f"{hadron_tag}, {frame}, {itd_q_tag}, {itd_gevp_tag}, reference {itd_ref_used}, E_ref = {E_ref:.4f}")
print("ratio fit:", os.path.basename(run_fits["ratio_fit_file"]))
print("2pt fit:  ", os.path.basename(run_fits["pt2_fit_file"]))
print(f"{'pz':>3s} {'E':>8s} {'E_ref/E':>8s} {f'S(w={w_values[0]})':>11s} {f'S(w={w_values[4]})':>11s} {f'S(w={w_values[8]})':>11s} {f'D(w={w_values[4]})':>11s} {f'D(w={w_values[8]})':>11s}")
for ipz, pz in enumerate(pz_values):
    print(f"{pz:3d} {E_central[ipz]:8.4f} {E_ref / E_central[ipz]:8.4f} " + " ".join(f"{str(gv.gvar(a, b)):>11s}" for a, b in [(single_mean[0, ipz], single_err[0, ipz]), (single_mean[4, ipz], single_err[4, ipz]), (single_mean[8, ipz], single_err[8, ipz]), (double_mean[4, ipz], double_err[4, ipz]), (double_mean[8, ipz], double_err[8, ipz])]))


#---- save
itd_name = "itd_" + os.path.basename(run_fits["ratio_fit_file"])[len("ratio_fit_"):-len(f"_{hadron_tag}.h5")] + f"_t{run_fits['pt2_fit_range']}_ref{itd_ref_used}_wref{itd_w_ref}_{hadron_tag}"
itd_file = os.path.join(itd_root, "itd_data", itd_subfolder, itd_name + ".h5")
os.makedirs(os.path.dirname(itd_file), exist_ok=True)
with h5py.File(itd_file, "w") as f, h5py.File(run_fits["ratio_fit_file"], "r") as fit_in:
    f["single_ratio_jk"] = single_jk    #(jk, w, pz): M(p, w) / M_ref(w) * E_ref / E
    f["double_ratio_jk"] = double_jk    #(jk, w, pz): [M(p, w) / M(p, w_ref)] / [M_ref(w) / M_ref(w_ref)]
    f["nu"] = nu                        #(w, pz): 2 pi / Ls * pz * w
    f["E"] = E_central                  #(pz,): the central E0 of the 2pt fit
    f["E_ref"] = E_ref
    f["w_ref"] = itd_w_ref
    f["reference"] = itd_ref_used
    f["ratio_fit_file"] = run_fits["ratio_fit_file"]
    f["pt2_fit_file"] = run_fits["pt2_fit_file"]
    f["ref_ratio_fit_file"] = ref_fits["ratio_fit_file"]
    f["ref_pt2_fit_file"] = ref_fits["pt2_fit_file"]
    f["pt2_fit_range_tag"] = run_fits["pt2_fit_range"]
    for name in ["w_list", "pz_list", "cfg_list", "q_list", "hyp", "model", "fit_half", "fit_window", "fit_range_tag", "phase_attached_to",
                 "gevp_tag", "gevp_basis", "op_basis", "smear_names", "smear_list", "gamma_names", "t0", "t_ref"]:
        fit_in.copy(name, f)            #what the ratio fit used
print("saved", itd_file)


#---- plot: the single and the double ratio versus nu, one line per pz
itd_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
itd_markers = ["o", "s", "^", "D", "v", "P", "X"]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, mean, err, title in [(axes[0], single_mean, single_err, "single ratio:  M(p, w) / M_ref(w) x E_ref / E"),
                             (axes[1], double_mean, double_err, f"double ratio:  [M(p, w) / M(p, {itd_w_ref})] / [M_ref(w) / M_ref({itd_w_ref})]")]:
    mean = np.where(err < itd_plot_err_max, mean, np.nan)
    for ipz, pz in enumerate(pz_values):
        ax.errorbar(nu[:, ipz], mean[:, ipz], err[:, ipz], color=itd_colors[ipz], marker=itd_markers[ipz], ms=5, lw=0.8, elinewidth=1, capsize=2, label=f"pz = {pz}")
    ax.axhline(1, color="#52514e", lw=0.8, ls=":")
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(r"$\nu = P_z z$")
    ax.set_ylabel("Re")
    ax.set_title(title, fontsize=10)
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
fig.suptitle(f"{hadron_tag}, {frame}, {itd_q_tag}, reference {itd_ref_used}, {len(run_fits['cfg_list'])} cfg\n{' '.join(itd_basis)}, t0 {itd_t0}, t_ref {itd_t_ref}", fontsize=10)
fig.tight_layout()
itd_plot_file = os.path.join(itd_root, "itd_plots", itd_subfolder, itd_name + ".png")
os.makedirs(os.path.dirname(itd_plot_file), exist_ok=True)
fig.savefig(itd_plot_file, dpi=150)
plt.show()
print("saved", itd_plot_file)